# Transformer (STFTnet-inspired) Gesture Classifier

## Overview

This notebook implements a **Transformer-based dynamic gesture recognition** model for data-glove time-series data, inspired by the architecture described in:

> *"A Convolutional-Transformer-Based Approach for Dynamic Gesture Recognition of Data Gloves"* (STFTnet), which achieved **99.72% accuracy** on their benchmark dataset.

---

## Why Transformers for Gesture Sequences?

Each glove recording is a **multivariate time series** — hundreds of IMU and flex sensor channels sampled over the duration of a gesture. The fundamental modelling challenge is relating sensor measurements across time to recognise the gesture.

### The problem with LSTMs (and RNNs generally)

LSTMs process sequences **step-by-step**: the hidden state at step *t* is computed from the hidden state at step *t-1*. To relate timestep 1 to timestep 155, the information must survive **154 sequential hidden-state updates**. Each update applies a learned transformation — multiplying the hidden state by weight matrices and passing it through non-linearities. Every such operation risks distorting or forgetting the early information. This is the **long-range dependency problem**.

Mathematically, gradients flowing back through 154 steps are multiplied together 154 times. If those multiplied factors are slightly less than 1 (which happens routinely), the gradient shrinks exponentially — the **vanishing gradient** problem you saw in your backpropagation lectures.

### What a Transformer does differently

Instead of processing sequentially, the Transformer **looks at the entire sequence at once**. A mechanism called **self-attention** lets every timestep directly compare itself to every other timestep in a single matrix operation — no sequential bottleneck.

Timestep 80 can directly attend to timestep 10 and timestep 150 in one step. For a gesture like a 'wave', the model can directly relate the hand's orientation at the beginning to its position at the end, without depending on a fragile chain of hidden states.

---

## Architecture Pipeline (with shapes)

```
Input  (batch, 156, C)          ← C = number of sensor channels (e.g. 180)
       │
       ▼
[MFABlock]  (batch, 156, C)     ← channel-attention gate: downweights noisy sensors
       │
       ▼
[DepthwiseSepConv1D]  (batch, 156, D_MODEL)   ← embed C → D_MODEL=64 dimensions
       │
       ▼
[PositionalEncoding]  (batch, 156, D_MODEL)   ← add sinusoidal position signal
       │
       ▼
[TransformerEncoderBlock × N_LAYERS=3]        ← each block: MHA + FFN + residuals
       │                (batch, 156, D_MODEL) ← shape unchanged through each block
       ▼
[GlobalAveragePooling1D]  (batch, D_MODEL)    ← aggregate all 156 timestep vectors
       │
       ▼
[Dense(64, relu) → Dropout]  (batch, 64)
       │
       ▼
[Dense(NUM_CLASSES, softmax)]  (batch, 4)     ← 4-class gesture probabilities
```

---

## Key Hyperparameters (from paper)

| Parameter | Paper | This notebook |
|---|---|---|
| `D_MODEL` | 64 | 64 |
| `N_LAYERS` | 3 | 3 |
| `N_HEADS` | 16 | 4 (auto-adjusts to feature count) |
| `MFA_GAMMA` | 256 | 64 (scaled to sensor count) |
| `LR` | 0.001 | 0.001 |
| `batch_size` | 64 | 32 |
| `weight_decay` | 1e-8 | 1e-8 (AdamW) |
| Sequence length | 50 steps | 156 steps |

---

## Cell 1 — Install Dependencies

This cell ensures all required Python packages are available in the Jupyter environment.
Run it once at the start of each session; subsequent cells import from these packages.

In [ ]:
"""
Install all Python packages required by this notebook.

Packages used:
  tensorflow    — deep learning framework providing Keras, layers, optimisers
  scikit-learn  — train/val/test split, classification report, confusion matrix
  matplotlib    — plotting learning curves and attention heatmaps
  seaborn       — heatmap styling for confusion matrix
  pandas        — DataFrame operations (class-balance inspection)
  numpy         — array maths (positional encoding, metric computation)
  scipy         — signal filtering and resampling in the data loader utility

tensorflow-addons provides AdamW (Adam + weight decay), which the STFTnet paper
uses to regularise the model. If unavailable for this TF version, we fall back
to standard Adam (weight_decay is then not applied, noted in the config JSON).
"""
import subprocess, sys

def pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pkg], check=True)

pip_install("tensorflow")
pip_install("scikit-learn")
pip_install("matplotlib")
pip_install("seaborn")
pip_install("pandas")
pip_install("numpy")
pip_install("scipy")

# Optional: tensorflow-addons for AdamW (may not be available for all TF versions)
try:
    pip_install("tensorflow-addons")
    print("tensorflow-addons installed.")
except Exception:
    print("tensorflow-addons unavailable — will use Adam with weight_decay note.")

print("Installation complete.")

---

## Cell 2 — Imports and Setup

All library imports live here so they are easy to find and the namespace is
established before any other cell runs.

Key imports explained:
- `tensorflow` / `keras` — the neural network framework; `K` (backend) is used
  to programmatically set the learning rate during warmup
- `numpy` — used for positional encoding maths and metric arrays
- `sklearn` — metrics and the stratified train/val/test split
- `matplotlib` / `seaborn` — visualising learning curves and confusion matrices
- `gridspec` — allows a mixed-layout figure for the attention visualisation
- `Path` — clean cross-platform file path handling
- Local `utils.*` — the shared data loading pipeline written for this project

In [ ]:
"""
Import all libraries and configure global plot aesthetics.

Everything that is imported here is used in subsequent cells.
Reading this cell gives a complete map of what this notebook depends on.
"""
import sys, os

# ─── Make the project root importable ─────────────────────────────────────────
# The notebook lives in  ThesisRepo/ML/;  the utils/ package lives in
# ThesisRepo/utils/.  We add the parent directory to Python's module search
# path so that  `from utils.data_loader import ...`  works regardless of
# where Jupyter was launched from.
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

# ─── Numeric and data libraries ───────────────────────────────────────────────
import numpy  as np
import pandas as pd
import json
from pathlib import Path

# ─── Visualisation ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec   # flexible subplot grid (attention viz)
import seaborn as sns

# ─── TensorFlow / Keras ───────────────────────────────────────────────────────
import tensorflow as tf
import tensorflow.keras as keras
import tensorflow.keras.backend as K    # used to set LR on each warmup step

# ─── sklearn metrics ──────────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

# ─── Optional: tensorflow-addons for AdamW ────────────────────────────────────
try:
    import tensorflow_addons as tfa
    TFA_AVAILABLE = True
except ImportError:
    TFA_AVAILABLE = False

# ─── Local data-loader utilities ─────────────────────────────────────────────
# build_dataset  : discovers CSVs, loads, filters, resamples, normalises → (X, y)
# split_dataset  : stratified train / val / test split
from utils.data_loader import build_dataset, split_dataset

# ─── Plot style ───────────────────────────────────────────────────────────────
# Consistent colour palette across all plots in this notebook.
# TEAL and RUST are used for train vs. validation curves respectively.
plt.rcParams.update({
    "figure.dpi":    120,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.size":     11,
})
TEAL      = "#2a9d8f"
RUST      = "#e76f51"
DARK_TEAL = "#1a6b5f"

print(f"TensorFlow version : {tf.__version__}")
print(f"Keras version      : {keras.__version__}")
print(f"NumPy version      : {np.__version__}")
print(f"AdamW available    : {TFA_AVAILABLE}")

---

## Cell 3 — Configuration

**This is the single cell you edit to control the entire pipeline.**
Every hyperparameter, data path, and model setting is defined here.
Downstream cells read these variables — never hard-code values in later cells.

### Sensor selection flags
The glove has many sensor groups (finger IMUs at two locations each, flex sensors,
palm IMU, wrist IMU). The `USE_*` flags let you toggle entire sensor groups on or
off without touching any other code. The column list is computed automatically from
these flags in the next cell.

### Model hyperparameters
| Symbol | Meaning |
|---|---|
| `D_MODEL` | Dimensionality of each timestep's embedding inside the Transformer (the "width" of every layer) |
| `N_LAYERS` | How many Transformer encoder blocks to stack |
| `N_HEADS` | Number of parallel attention heads per block |
| `FFN_DIM` | Hidden dimension of the Feed-Forward Network inside each block (typically 2–4× D_MODEL) |
| `MFA_GAMMA` | Expansion factor inside the MFA channel-attention bottleneck |
| `LR_WARMUP_STEPS` | Number of gradient steps over which the learning rate ramps up linearly |

In [ ]:
"""
CONFIGURATION CELL — edit this cell to control every aspect of the pipeline.

All variables defined here are CAPITALS to signal that they are global
configuration constants. Later cells import from this namespace; changing
a value here propagates automatically to every subsequent cell.
"""

# =============================================================================
# DATA
# =============================================================================
DATA_ROOT = '../data'   # Root folder containing the 4 gesture label subfolders:
                        #   TwoHand_L_Fist_R_Fist_filtered_butterworth_lp/
                        #   TwoHand_L_Flat_R_Flat_filtered_butterworth_lp/
                        #   TwoHand_L_Okay_R_Okay_filtered_butterworth_lp/
                        #   TwoHand_L_Rock_R_Rock_filtered_butterworth_lp/

FS_HZ      = 31.0       # Confirmed sampling rate (~156 rows / 5 s ≈ 31.2 Hz)
TARGET_LEN = 156        # Native timesteps per trial (5 s × ~31 Hz).
                        # Reduce (e.g. to 78) to halve resolution and speed up training.

# =============================================================================
# COLUMN / SENSOR SELECTION
# =============================================================================
# Toggle entire sensor groups on / off — the feature column list is assembled
# automatically in the next cell.  All combinations are valid.

USE_LEFT_HAND  = True    # Include left-hand sensors
USE_RIGHT_HAND = True    # Include right-hand sensors

# Per-finger IMU locations
USE_DISTAL_IMU   = True  # 'mid'  — distal phalanx IMU  (fingertip segment)
USE_PROXIMAL_IMU = True  # 'prox' — proximal phalanx IMU (base segment, near palm)

# IMU channel types (applies to finger IMUs, palm_prox, and wrist)
USE_ACCELEROMETER = True  # ax, ay, az  — linear acceleration in 3D (units: g)
USE_ORIENTATION   = True  # yaw, pitch, roll  (wrist uses 'heading' instead of 'yaw')

# Flex sensors — measure how bent each finger joint is
USE_FLEX_SENSORS  = True  # {hand}_{finger}_{mcp_flex / pip_flex}
                           # NOTE: palm flex is always -1 (no sensor) → auto-excluded

# Back-of-hand IMU (palm_prox — the real palm IMU;
#                   palm_mid is always 0 → auto-excluded by the data loader)
USE_PALM_IMU  = True

# Wrist IMU  ({hand}_wrist_{ax/ay/az/heading/pitch/roll})
USE_WRIST_IMU = True

# =============================================================================
# PREPROCESSING
# =============================================================================
# Temporal filter applied to each sensor channel before training.
# The data files are already Butterworth LP-filtered at 5 Hz by the glove
# firmware; 'none' uses them as-is, which is the recommended starting point.
#
# Options: 'none' | 'butterworth_lp' | 'butterworth_hp' | 'butterworth_bp'
#          'moving_average' | 'savgol' | 'median'
FILTER_TYPE = 'none'

# Normalisation method applied after resampling.
# 'minmax' → scales each channel to [0, 1]
# 'zscore' → zero-mean, unit-variance per channel
# 'none'   → skip normalisation (not recommended; raw IMU/flex scales differ)
NORMALIZATION   = 'minmax'
PER_SAMPLE_NORM = True   # True = normalise each trial independently (recommended)
                         # False = normalise globally across all trials

# =============================================================================
# DATASET SPLIT
# =============================================================================
TRAIN_RATIO = 0.70   # 70 % train   →  280 / 400 samples
VAL_RATIO   = 0.10   # 10 % val     →   40 / 400 samples
                     # 20 % test    →   80 / 400 samples  (auto = 1 - 0.70 - 0.10)
RANDOM_SEED = 42     # Seed for reproducibility; same seed gives same split every run

# =============================================================================
# MODEL HYPERPARAMETERS
# =============================================================================

# ── Multi-sensor Feature Attention block (MFA) ─────────────────────────────
USE_MFA_BLOCK = True
MFA_GAMMA     = 64     # Bottleneck expansion in MFA channel-attention (paper: 256)
                       # Smaller value → fewer parameters, faster training

# ── Embedding (Depthwise Separable Conv1D) ─────────────────────────────────
D_MODEL = 64           # Embedding dimension: every timestep becomes a 64-dim vector
                       # MUST be divisible by N_HEADS; auto-adjusted if needed

# ── Transformer encoder ────────────────────────────────────────────────────
N_LAYERS   = 3         # Number of stacked encoder blocks (deeper → more capacity)
N_HEADS    = 4         # Parallel attention heads per block.
                       # Each head uses d_k = D_MODEL / N_HEADS = 16 dimensions.
                       # Multiple heads let the model attend to different temporal
                       # relationships simultaneously.
FFN_DIM    = 128       # Feed-Forward Network hidden dim (2 × D_MODEL here)
TRANSFORMER_DROPOUT = 0.1   # Dropout inside attention and FFN sub-layers

USE_POSITIONAL_ENCODING = True   # Whether to add sinusoidal position embeddings
                                 # Set False to test the model without position info

# ── Dense classification head ──────────────────────────────────────────────
DENSE_UNITS  = [64]    # One hidden Dense(64, relu) layer before the output
DROPOUT_RATE = 0.3     # Output dropout after each head Dense layer

# =============================================================================
# TRAINING
# =============================================================================
EPOCHS              = 80
BATCH_SIZE          = 32
LEARNING_RATE       = 1e-3    # Base learning rate for Adam / AdamW
WEIGHT_DECAY        = 1e-8    # L2 weight regularisation strength (AdamW only)
EARLY_STOP_PATIENCE = 15      # Stop training if val_accuracy doesn't improve for
                               # this many consecutive epochs
LR_WARMUP_STEPS     = 50      # Linear LR warmup: start near 0, ramp to LEARNING_RATE
                               # over this many gradient steps. Prevents large weight
                               # updates during the first epoch when all weights are
                               # random and representations are unstable.

---

## Cell 4 — Resolve Sensor Selection → Column Names

The `USE_*` flags in the CONFIG cell are human-readable booleans. This cell
converts them into the concrete list of CSV column names that will be passed
to the data loader.

`build_column_groups()` (from `utils/data_loader.py`) knows the full column
schema of the glove CSV files and assembles the list based on your flags.
The printed output lets you verify exactly which sensor channels are included.

In [ ]:
"""
Resolve the CONFIG sensor-selection flags into a concrete list of CSV column names.

The data_loader utility knows the full column schema of the glove CSV files.
build_column_groups() returns an ordered list of strings like:
    ['right_palm_prox_yaw', 'right_palm_prox_pitch', ..., 'left_wrist_roll']

This list is passed to build_dataset() and determines:
  - Which columns are loaded from each CSV file
  - N_FEATURES = len(feature_cols) — the width of each timestep vector fed to the model
"""
from utils.data_loader import (
    build_column_groups, HANDS, FINGERS, LOCS,
    IMU_ALL_CHANNELS, FLEX_CHANNELS, WRIST_ALL_CHANNELS,
)

# ─── Resolve which hands to include ──────────────────────────────────────────
# HANDS is the constant ['right', 'left'].  We filter it down based on the
# USE_LEFT_HAND / USE_RIGHT_HAND flags set in CONFIG.
# The `or HANDS` fallback means: if BOTH flags were accidentally set to False,
# include both hands anyway — preventing an empty feature list.
_hands = ([h for h in HANDS if (USE_LEFT_HAND  and h == "left")
                              or (USE_RIGHT_HAND and h == "right")]
          or HANDS)

# ─── All fingers are always included ─────────────────────────────────────────
# Only the IMU LOCATION per finger (distal tip vs. proximal base) is configurable.
_fingers = FINGERS     # ['thumb', 'index', 'middle', 'ring', 'pinky']

# ─── Resolve which finger IMU locations to include ────────────────────────────
# 'mid'  = distal phalanx   (sensor at the fingertip segment)
# 'prox' = proximal phalanx (sensor at the base segment, nearest the palm)
# Same `or LOCS` fallback: if both flags were False, include both.
_locs = ([l for l in LOCS
          if (USE_DISTAL_IMU   and l == "mid")
          or (USE_PROXIMAL_IMU and l == "prox")]
         or LOCS)

# ─── Assemble the full column-name list ──────────────────────────────────────
# build_column_groups() returns a flat, ordered list of column name strings.
# The ordering matches the physical layout of the glove (right → left, palm →
# fingers → wrist) which helps the model learn spatial relationships.
feature_cols = build_column_groups(
    hands              = _hands,
    fingers            = _fingers,
    locs               = _locs,
    include_flex       = USE_FLEX_SENSORS,    # MCP and PIP joint bend sensors
    include_palm_prox  = USE_PALM_IMU,        # back-of-hand IMU
    include_wrist      = USE_WRIST_IMU,       # wrist IMU (6 channels)
    include_accel      = USE_ACCELEROMETER,   # ax, ay, az per IMU
    include_orient     = USE_ORIENTATION,     # yaw/pitch/roll per IMU
)

print(f"Feature columns selected: {len(feature_cols)}")
print()

# ─── Print a grouped summary for visual verification ─────────────────────────
# Groups columns by their sensor-group prefix (e.g. 'right_thumb_mid') so you
# can quickly confirm which sensors and channels are included.
groups = {}
for c in feature_cols:
    parts = c.split("_")
    grp   = "_".join(parts[:3]) if len(parts) >= 4 else "_".join(parts[:2])
    groups.setdefault(grp, []).append(c)

for grp, cols in list(groups.items())[:12]:
    print(f"  {grp:40s}  ({len(cols)} cols): {[c.split('_')[-1] for c in cols]}")
if len(groups) > 12:
    print(f"  ... and {len(groups)-12} more groups")

---

## Cell 5 — Load and Preprocess Dataset

`build_dataset()` orchestrates the full preprocessing pipeline:

1. **Discover** all CSV trial files under `DATA_ROOT`, infer the gesture label from the subfolder name
2. **Load** each CSV, keeping only the `feature_cols` columns (dropping metadata and always-zero sensors)
3. **Filter** each sensor channel through `FILTER_TYPE` (optional temporal smoothing)
4. **Resample** each trial to exactly `TARGET_LEN` timesteps using linear interpolation
5. **Stack** all trials into a 3-D NumPy array `X` of shape `(N, TARGET_LEN, C)`
6. **Normalise** using `NORMALIZATION` (per-sample MinMax by default)

The result is `X` and `y` — the input matrix and label vector for supervised learning,
exactly as described in your COMP3308 lectures.

In [ ]:
"""
Load all gesture trial CSVs, apply the preprocessing pipeline, and store
the resulting arrays in X, y, labels, and CLASS_NAMES.

After this cell:
  X      — shape (N, TARGET_LEN, C)  float32 — the model's input data
             N = number of trials (e.g. 400 across 4 gestures × 100 recordings each)
             TARGET_LEN = timesteps per trial (e.g. 156 at ~31 Hz × 5 s)
             C = len(feature_cols) — number of sensor channels included
  y      — shape (N,)                int64   — integer class label (0, 1, 2, or 3)
  labels — list of 4 gesture names in label-index order (e.g. ['Fist','Flat','Okay','Rock'])
"""

X, y, labels, feature_cols = build_dataset(
    data_root       = DATA_ROOT,
    feature_cols    = feature_cols,    # from Cell 4
    filter_type     = FILTER_TYPE,
    fs              = FS_HZ,
    target_len      = TARGET_LEN,
    normalization   = NORMALIZATION,
    per_sample_norm = PER_SAMPLE_NORM,
)

# ─── Derived constants ────────────────────────────────────────────────────────
# These are computed from the loaded data rather than hard-coded, so they
# automatically adapt if you change TARGET_LEN or the feature_cols selection.
N_SAMPLES, N_TIMESTEPS, N_FEATURES = X.shape
N_CLASSES  = len(labels)
CLASS_NAMES = labels    # human-readable class names for confusion matrix axes

print(f"\nDataset summary:")
print(f"  X shape    : {X.shape}   (samples, timesteps, features)")
print(f"  y shape    : {y.shape}   (integer class index per sample)")
print(f"  N_CLASSES  : {N_CLASSES}")
print(f"  Classes    : {CLASS_NAMES}")
print()

# ─── Verify class balance ────────────────────────────────────────────────────
# Unbalanced classes can bias accuracy metrics (a model that always predicts
# the majority class can achieve high accuracy without learning anything useful).
# With 100 samples per class this dataset is perfectly balanced.
print("Class distribution:")
for i, name in enumerate(labels):
    count = int((y == i).sum())
    bar   = "█" * (count // 5)
    print(f"  [{i}] {name:>45s}  {count:4d}  {bar}")

---

## Cell 6 — Train / Val / Test Split

Splitting the dataset into three non-overlapping subsets:

| Split | Fraction | Purpose |
|---|---|---|
| **Train** | 70% (280 samples) | Model weights are updated on these examples |
| **Val** | 10% (40 samples) | Monitors generalisation during training; used by EarlyStopping |
| **Test** | 20% (80 samples) | Held back entirely; gives the final unbiased accuracy |

The split is **stratified** — each subset maintains the same class proportions as the
full dataset. With 100 samples per class, each class appears in the expected 70/10/20
ratio. Stratification is described in your COMP3308 lectures as "an improvement of
the holdout method" that reduces the variance of the accuracy estimate.

The bar charts below let you verify that no class is accidentally over- or under-represented.

In [ ]:
"""
Stratified train / val / test split.

Uses sklearn's train_test_split with stratify=y to ensure each subset
has the same class distribution as the full dataset.

After this cell:
  X_train / y_train  — (280, 156, C) / (280,)  used to update model weights
  X_val   / y_val    — ( 40, 156, C) / ( 40,)  used by EarlyStopping
  X_test  / y_test   — ( 80, 156, C) / ( 80,)  held out until final evaluation
"""

# split_dataset() performs two train_test_split calls internally:
#   First:  separate test set (test_ratio = 1 - TRAIN_RATIO - VAL_RATIO = 0.20)
#   Second: split the remainder into train and val
(X_train, y_train), (X_val, y_val), (X_test, y_test) = split_dataset(
    X, y,
    train_ratio = TRAIN_RATIO,
    val_ratio   = VAL_RATIO,
    seed        = RANDOM_SEED,
)

print(f"\nTrain : {X_train.shape}  |  Val : {X_val.shape}  |  Test : {X_test.shape}")

# ── Verify class balance in each split ────────────────────────────────────────
# Plot bar charts of class counts across the three splits.
# Ideally all three plots should look nearly identical (balanced stratification).
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=False)
splits    = [(y_train, 'Train', TEAL), (y_val, 'Val', RUST), (y_test, 'Test', DARK_TEAL)]

for ax, (y_split, split_name, color) in zip(axes, splits):
    cnts = pd.Series(y_split).value_counts().sort_index()
    ax.bar(range(len(cnts)), cnts.values, color=color, edgecolor='white')
    ax.set_title(f'{split_name} ({len(y_split)} samples)', fontweight='bold')
    ax.set_xlabel('Class index')
    ax.set_ylabel('Count')
    ax.set_xticks(range(len(cnts)))
    ax.set_xticklabels(range(len(cnts)), fontsize=7)

plt.suptitle('Class Distribution per Split', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## Cell 7 — Model Definition

The model is built using the **TensorFlow/Keras functional API** with four custom
layer classes. All the novel concepts (self-attention, positional encoding, etc.)
are implemented here.

---

### Concept 1: Self-Attention — how a Transformer "reads" a sequence

Imagine the 156 timesteps as **156 students in a class**. Each student (timestep)
asks three questions:

| Symbol | Role | Question |
|---|---|---|
| **Q** (Query) | What am I looking for? | "What information do I need to understand my context?" |
| **K** (Key) | What do I advertise? | "What information do I contain?" (broadcast to all others) |
| **V** (Value) | What do I send? | "If someone attends to me, this is what they receive" |

For every pair of timesteps *i* and *j*, the **attention score** is:

```
score(i, j) = dot(Q_i, K_j) / sqrt(d_k)
```

- High score → student *i* finds student *j*'s information relevant
- Divided by √d_k (where d_k = D_MODEL / N_HEADS) to prevent the dot products
  from growing so large that softmax becomes saturated (all weight on one timestep,
  gradient ≈ 0 everywhere else — effectively a vanishing gradient problem again)

Normalise scores across all *j* for each *i*:
```
α(i, j) = softmax_j( score(i, j) )   ← weights sum to 1 across all j
```

Compute the output for timestep *i* as a weighted sum of all Values:
```
output_i = Σ_j  α(i, j) · V_j
```

This is **fully parallelisable** — all 156×156 scores are computed in one matrix
multiplication: `Attention(Q, K, V) = softmax( Q Kᵀ / √d_k ) V`.

---

### Concept 2: Multi-Head Attention

Instead of one attention operation, we run **N_HEADS = 4** independent attention
"heads" in parallel, each with smaller dimension d_k = D_MODEL / N_HEADS = 16.

Each head learns to attend to **different types of relationships**:
- Head 1 might learn: *"attend to the previous timestep"* (local motion continuity)
- Head 2 might learn: *"attend to the gesture's opening position"* (global context)
- Head 3 might learn: *"attend to moments with similar finger curvature"* (posture similarity)

Their outputs are **concatenated** (giving back D_MODEL dimensions) then linearly
projected, combining all the specialised views into one rich representation.

---

### Concept 3: Transformer Encoder Block structure

One block applies two sub-layers, each wrapped in a **residual connection**:

```
x' = LayerNorm( x + MultiHeadAttention(x) )
x''= LayerNorm( x' + FFN(x') )
```

**Residual connections** (the `x +` part) are the same idea as ResNet shortcut
connections you learned about with CNNs. From your backpropagation lectures:
gradients are multiplied through each layer during backprop. In a deep network,
these products can shrink to near-zero (vanishing gradient). The residual shortcut
provides a **direct path for gradients** — they can flow back without passing through
any learned transformation, keeping gradients healthy regardless of depth.

**LayerNorm** normalises the feature values across the D_MODEL dimension for each
timestep independently. Unlike BatchNorm (which normalises across the batch dimension),
LayerNorm works on each sample individually — important for sequence models where
each timestep's statistics may differ from others.

**FFN (Feed-Forward Network)**: two Dense layers applied independently to each timestep:
```
Dense(FFN_DIM=128, relu) → Dense(D_MODEL=64)
```
This is a standard 2-layer MLP — exactly the kind you built in COMP3308 — applied
position-wise. It adds non-linearity and increases the model's representational capacity
without mixing information across timesteps (that's the attention mechanism's job).

---

### Concept 4: MFA Block (Multi-sensor Feature Attention)

Before the Transformer, we apply a **channel-attention gate**:

```
x: (batch, 156, C)
   → GlobalAveragePooling1D  → (batch, C)     collapse time, one value per sensor
   → Dense(MFA_GAMMA, relu)  → (batch, γ)     learn inter-sensor relationships
   → Dense(C, sigmoid)       → (batch, C)     output α_c ∈ [0,1] per sensor channel
   → expand_dims + Multiply  → (batch, 156, C) scale each sensor by its learned weight
```

This asks: *"which sensors are most informative for distinguishing gestures?"*
Noisy or irrelevant sensor channels get a weight close to 0 (suppressed); highly
discriminative channels get a weight close to 1 (amplified). This is analogous
to Squeeze-and-Excitation (SE) networks from the CNN literature.

---

### Concept 5: Positional Encoding

Self-attention is a **set operation** — it has no built-in sense of order.
If you shuffled the timesteps, the same attention scores would be computed (just
reassigned to different positions). So the Transformer cannot tell whether
timestep 5 came before or after timestep 50 unless we explicitly tell it.

We add a **sinusoidal positional encoding** to each timestep's embedding before
the attention layers:

```
PE(t, 2i)   = sin( t / 10000^(2i / D_MODEL) )
PE(t, 2i+1) = cos( t / 10000^(2i / D_MODEL) )
```

- *t* is the timestep index (0 to 155)
- *i* is the dimension index (0 to D_MODEL/2 - 1)
- Low-*i* dimensions oscillate **quickly** → encode fine-grained position
- High-*i* dimensions oscillate **slowly** → encode coarse/global position

This creates a unique fingerprint for every position. Adding it directly to the
embedding means the attention mechanism can learn to use position information when
computing attention scores.

---

### Concept 6: Depthwise Separable Convolution (embedding layer)

Before the Transformer, we project the raw C sensor channels to D_MODEL = 64
dimensions. A standard `Dense` projection at each timestep is equivalent to a
1×1 convolution. We use **depthwise separable convolution** instead:

1. **Depthwise conv**: apply one separate filter per input channel → captures
   **local temporal patterns** within each sensor independently
2. **Pointwise conv** (1×1): mix the channels → creates the D_MODEL-dim embedding

This has far fewer parameters than a full Conv1D while being equally expressive.
You can think of it as factorising the spatial and channel operations — the same
idea as depth-wise separable convolutions in MobileNet.

In [ ]:
"""
Define all custom Keras layers and the full model builder function.

Classes defined here:
  MFABlock                  — Multi-sensor Feature Attention (channel gate)
  DepthwiseSeparableConv1D  — Efficient embedding: C channels → D_MODEL dims
  PositionalEncoding         — Sinusoidal timestep position injection
  TransformerEncoderBlock   — MHA + residual + LayerNorm + FFN + residual + LayerNorm

Function defined here:
  build_transformer_model() — assembles the full pipeline using Keras functional API

Shape flow through the full model:
  Input               : (batch, 156, C)         e.g. (32, 156, 180)
  After MFABlock      : (batch, 156, C)         same shape, but channels are scaled
  After DW-Sep-Conv   : (batch, 156, D_MODEL)   C → 64 dimensional embedding
  After PosEnc        : (batch, 156, D_MODEL)   position information added
  After Transformer×3 : (batch, 156, D_MODEL)   3 encoder blocks, shape unchanged
  After GAP           : (batch, D_MODEL)        temporal aggregation → one vector/sample
  After Dense head    : (batch, 64)             classification head
  Output (Softmax)    : (batch, NUM_CLASSES)    probability per gesture class
"""

# ─────────────────────────────────────────────────────────────────────────────
# Auto-adjust N_HEADS so D_MODEL is evenly divisible
# ─────────────────────────────────────────────────────────────────────────────
def find_valid_n_heads(d_model, requested_heads):
    """
    Return the largest divisor of d_model that is <= requested_heads.

    MultiHeadAttention requires D_MODEL to be evenly divisible by N_HEADS
    because each head is allocated d_k = D_MODEL // N_HEADS dimensions.
    If the user-configured N_HEADS doesn't divide D_MODEL, we auto-correct
    rather than crashing at runtime.

    Example: D_MODEL=64, N_HEADS=5 → tries 5, 4 (64%4==0) → returns 4.
    """
    if d_model % requested_heads == 0:
        return requested_heads
    for h in range(requested_heads, 0, -1):   # try smaller values until one divides evenly
        if d_model % h == 0:
            print(f"[WARNING] N_HEADS={requested_heads} does not divide D_MODEL={d_model}. "
                  f"Auto-adjusted N_HEADS → {h}")
            return h
    return 1  # fallback: single-head attention (always valid)

N_HEADS_ACTUAL = find_valid_n_heads(D_MODEL, N_HEADS)
print(f"D_MODEL={D_MODEL}, N_HEADS requested={N_HEADS}, N_HEADS used={N_HEADS_ACTUAL}")
print(f"→ d_k per head = {D_MODEL // N_HEADS_ACTUAL}  (dimension of each head's Q/K/V)")


# ─────────────────────────────────────────────────────────────────────────────
# 1.  MFABlock — Multi-sensor Feature Attention
# ─────────────────────────────────────────────────────────────────────────────
class MFABlock(keras.layers.Layer):
    """
    Multi-sensor Feature Attention Block.

    Implements a channel-attention (Squeeze-and-Excitation style) mechanism
    specifically for multi-channel sensor time series:

      Step 1 — Squeeze (GlobalAveragePooling1D):
          Collapse the time dimension by averaging over all T timesteps.
          Input  (batch, T, C)  →  (batch, C)
          This produces one scalar per sensor channel — a summary of how
          active that sensor was over the entire gesture recording.

      Step 2 — Excitation (two Dense layers forming a bottleneck):
          Dense(gamma, relu)   (batch, C) → (batch, gamma)
              Expand to a higher-dimensional space to learn cross-sensor
              interactions (which sensors are correlated, which are redundant).
          Dense(C, sigmoid)    (batch, gamma) → (batch, C)
              Compress back to a weight per sensor; sigmoid squashes to [0,1]
              so each weight is interpretable as "how much of this sensor to keep".

      Step 3 — Scale (Multiply):
          Expand the weight to (batch, 1, C) so it broadcasts across time.
          Multiply with the original input: (batch, T, C) * (batch, 1, C)
          → each sensor channel is uniformly scaled up or down across all timesteps.

    The block learns, through backpropagation, which sensor channels are most
    discriminative for gesture recognition and amplifies them while suppressing
    noisy or redundant ones.
    """

    def __init__(self, gamma, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma   # expansion factor for the bottleneck Dense layer

    def build(self, input_shape):
        n_features = input_shape[-1]   # C — number of sensor channels

        # GlobalAveragePooling1D: collapses the T dimension → (batch, C)
        self.gap  = keras.layers.GlobalAveragePooling1D()

        # Bottleneck expansion: learns cross-channel relationships
        self.fc1  = keras.layers.Dense(self.gamma, activation='relu',
                                       name='mfa_fc1')   # (batch, gamma)

        # Gate output: one weight per channel in [0, 1]
        self.fc2  = keras.layers.Dense(n_features, activation='sigmoid',
                                       name='mfa_fc2')   # (batch, C)

        # Element-wise multiply (handles broadcasting automatically in call())
        self.mul  = keras.layers.Multiply()
        super().build(input_shape)

    def call(self, x):
        # x: (batch, T, C)
        gap  = self.gap(x)               # (batch, C) — squeeze: global time average
        gate = self.fc1(gap)             # (batch, gamma) — excitation: expand
        gate = self.fc2(gate)            # (batch, C) — excitation: gate weights

        # Expand the gate so it can broadcast over the time dimension:
        # (batch, C) → (batch, 1, C) → multiplied with (batch, T, C)
        # Broadcasting: the 1 expands to match T, scaling each timestep identically.
        gate = tf.expand_dims(gate, axis=1)   # (batch, 1, C)
        return x * gate                        # (batch, T, C) — channel-gated output

    def get_config(self):
        # Required so the model can be saved and reloaded with custom_objects
        config = super().get_config()
        config.update({'gamma': self.gamma})
        return config


# ─────────────────────────────────────────────────────────────────────────────
# 2.  DepthwiseSeparableConv1D — Local temporal embedding → D_MODEL
# ─────────────────────────────────────────────────────────────────────────────
class DepthwiseSeparableConv1D(keras.layers.Layer):
    """
    Depthwise Separable Conv1D embedding layer.

    Projects sensor data from C input channels to D_MODEL dimensions using
    a factorised convolution that is computationally cheaper than standard Conv1D.

    How it works:
    ─────────────
    Standard Conv1D (for comparison): one filter of size (kernel_size, C) per output
    channel. All input channels are mixed together by every filter simultaneously.
    Parameters: kernel_size × C × D_MODEL

    Depthwise Separable factorises this into two steps:

    Step 1 — Depthwise Convolution:
        Apply one separate filter (size: kernel_size × 1) to each input channel
        independently. Each filter only "sees" its own channel — it captures
        LOCAL TEMPORAL PATTERNS within that one sensor.
        Input  (batch, T, C)  →  (batch, T, C)  (same number of channels)
        Parameters: kernel_size × C  (much fewer than standard)

        Implementation note: Keras doesn't have DepthwiseConv1D, so we use
        DepthwiseConv2D with a (1, kernel_size) kernel and squeeze the extra dim.

    Step 2 — Pointwise Convolution (1×1 Conv):
        Apply D_MODEL filters of size 1×1, which MIX the channels together.
        Projects C → D_MODEL.
        Input  (batch, T, C)  →  (batch, T, D_MODEL)
        Parameters: C × D_MODEL

    Total parameters: kernel_size×C + C×D_MODEL  vs  kernel_size×C×D_MODEL (standard)
    For kernel_size=3, C=180, D_MODEL=64: 3×180 + 180×64 = 540+11520=12060
    vs standard: 3×180×64 = 34560  →  ~65% fewer parameters.

    The batch normalisation after the projection stabilises the activations before
    they enter the Transformer (prevents very large or very small embedding values
    that could destabilise attention scores).
    """

    def __init__(self, d_model, kernel_size=3, **kwargs):
        super().__init__(**kwargs)
        self.d_model     = d_model       # target embedding dimension
        self.kernel_size = kernel_size   # temporal window for the depthwise filter

    def build(self, input_shape):
        # ── Depthwise convolution ─────────────────────────────────────────────
        # DepthwiseConv2D expects 4-D input (batch, H, W, C).
        # We'll expand our (batch, T, C) to (batch, 1, T, C) in call().
        # Kernel shape (1, kernel_size): spans 1 in H and kernel_size in time (W).
        self.dw_conv = keras.layers.DepthwiseConv2D(
            kernel_size=(1, self.kernel_size),   # only convolves along time axis
            padding='same',                      # output length = input length
            activation='relu',
            name='dw_conv'
        )

        # ── Pointwise (1×1) convolution ───────────────────────────────────────
        # After depthwise, shape is still (batch, T, C).
        # Conv1D with kernel_size=1 mixes channels: C → d_model.
        self.pw_conv = keras.layers.Conv1D(
            filters=self.d_model,
            kernel_size=1,    # 1×1: no temporal mixing, only channel mixing
            padding='same',
            activation='relu',
            name='pw_conv'
        )

        # ── Batch Normalisation ───────────────────────────────────────────────
        # Normalises activations across the batch for each (time, feature) position.
        # Keeps the embedding values in a stable range before they enter attention.
        self.bn = keras.layers.BatchNormalization(name='dw_sep_bn')
        super().build(input_shape)

    def call(self, x, training=False):
        # x: (batch, T, C)

        # Expand to 4-D for DepthwiseConv2D
        x_4d = tf.expand_dims(x, axis=1)     # (batch, 1, T, C)
        x_dw = self.dw_conv(x_4d)            # (batch, 1, T, C) — depthwise convolution
        x_3d = tf.squeeze(x_dw, axis=1)      # (batch, T, C)    — squeeze back to 3-D

        # Pointwise projection: mixes channels, expands to D_MODEL
        x_pw = self.pw_conv(x_3d)            # (batch, T, D_MODEL)

        # Batch normalise the embedding
        return self.bn(x_pw, training=training)    # (batch, T, D_MODEL)

    def get_config(self):
        config = super().get_config()
        config.update({'d_model': self.d_model, 'kernel_size': self.kernel_size})
        return config


# ─────────────────────────────────────────────────────────────────────────────
# 3.  PositionalEncoding — Sinusoidal injection of position information
# ─────────────────────────────────────────────────────────────────────────────
class PositionalEncoding(keras.layers.Layer):
    """
    Sinusoidal positional encoding (Vaswani et al., 2017 — "Attention Is All You Need").

    WHY THIS IS NEEDED
    ------------------
    Self-attention is a SET operation: it computes scores between ALL pairs of
    timesteps simultaneously. If you shuffled the order of the 156 timesteps,
    the same attention scores would be computed (just rearranged). The Transformer
    has NO built-in sense of order — it treats the sequence like an unordered bag.

    To give it order awareness, we ADD a unique positional fingerprint to each
    timestep's embedding BEFORE it enters the attention layers.

    THE ENCODING FORMULA
    --------------------
    For timestep position t and embedding dimension index i:

        PE(t, 2i)   = sin( t / 10000^(2i / d_model) )
        PE(t, 2i+1) = cos( t / 10000^(2i / d_model) )

    Intuition:
    - Each dimension pair (2i, 2i+1) forms a sinusoidal wave at a specific frequency.
    - Dimension 0 oscillates at frequency 1 — it completes one full cycle across
      all 156 timesteps (coarse position encoding).
    - Dimension 62 oscillates much faster — it can distinguish nearby timesteps
      (fine-grained position encoding).
    - Together, the pattern is unique for every position t from 0 to max_len.

    This is DETERMINISTIC (not learned) — it's computed once at build time and
    stored as a constant tensor. Crucially, the model can learn to use the positional
    information by attending to specific combinations of dimensions.

    SHAPE FLOW
    ----------
    Input  x   : (batch, T, D_MODEL)
    pe slice   : (1, T, D_MODEL)       — broadcast over batch dimension
    Output     : (batch, T, D_MODEL)   — same shape, values shifted by position encoding
    """

    def __init__(self, max_len=512, **kwargs):
        super().__init__(**kwargs)
        self.max_len = max_len   # maximum sequence length this layer can handle

    def build(self, input_shape):
        d = input_shape[-1]     # D_MODEL — embedding dimension
        T = min(input_shape[-2] or self.max_len, self.max_len)

        # ── Compute the sinusoidal encoding table ─────────────────────────────
        # positions: column vector (T, 1) — the t values 0, 1, ..., T-1
        positions = np.arange(T)[:, np.newaxis]           # (T, 1)

        # dims: row vector (1, D_MODEL) — the i values 0, 1, ..., D_MODEL-1
        dims      = np.arange(d)[np.newaxis, :]           # (1, D_MODEL)

        # Compute the angle: t / 10000^(2*(i//2) / d_model)
        # Broadcasting: (T, 1) / (1, D_MODEL) → (T, D_MODEL)
        angles    = positions / np.power(
            10000.0, (2 * (dims // 2)) / np.float32(d)
        )   # (T, D_MODEL)

        # Apply sin to even-indexed dimensions, cos to odd-indexed dimensions
        angles[:, 0::2] = np.sin(angles[:, 0::2])   # even dims: sin
        angles[:, 1::2] = np.cos(angles[:, 1::2])   # odd  dims: cos

        # Store as a constant tensor with shape (1, T, D_MODEL)
        # The leading 1 allows broadcasting over the batch dimension.
        self.pe = tf.constant(angles[np.newaxis, :, :], dtype=tf.float32)
        super().build(input_shape)

    def call(self, x):
        # x: (batch, T, D_MODEL)
        # self.pe: (1, T_max, D_MODEL)
        # Slice to the actual sequence length (handles variable-length inputs)
        seq_len = tf.shape(x)[1]
        return x + self.pe[:, :seq_len, :]   # (batch, T, D_MODEL)

    def get_config(self):
        config = super().get_config()
        config.update({'max_len': self.max_len})
        return config


# ─────────────────────────────────────────────────────────────────────────────
# 4.  TransformerEncoderBlock
# ─────────────────────────────────────────────────────────────────────────────
class TransformerEncoderBlock(keras.layers.Layer):
    """
    Standard Transformer encoder block (post-norm variant).

    Architecture (two sub-layers, each with residual connection and LayerNorm):
    ─────────────────────────────────────────────────────────────────────────────
    Sub-layer 1 — Multi-Head Self-Attention:
        attn_out = MultiHeadAttention(x, x)   ← query=x, key=x, value=x (self-attention)
        x'  = LayerNorm( x + Dropout(attn_out) )

    Sub-layer 2 — Feed-Forward Network:
        ffn_out  = Dense(ffn_dim, relu)(x')   ← expand to ffn_dim dimensions
        ffn_out  = Dense(d_model)(ffn_out)    ← project back to d_model
        x'' = LayerNorm( x' + Dropout(ffn_out) )

    RESIDUAL CONNECTIONS (the `x +` part)
    --------------------------------------
    Every sub-layer's output is ADDED BACK to its input before LayerNorm.
    This is called a "shortcut connection" or "skip connection".

    Why does this matter? From your backpropagation lectures: in a deep network,
    the gradient of the loss w.r.t. an early layer's weights is the PRODUCT of
    many Jacobian matrices (one per layer). If these matrices have entries < 1,
    the product shrinks exponentially → vanishing gradient.

    The residual adds a direct path:
        gradient = (gradient through sub-layer) + (gradient through identity shortcut)
    The identity shortcut always contributes a gradient of 1.0, so even if the
    sub-layer's gradient is tiny, the total gradient remains healthy.
    This is the same principle as ResNet (2015) — a CNN architecture you studied.

    LAYER NORMALISATION
    -------------------
    After each residual addition, we normalise across the D_MODEL dimension:
        LN(x) = (x - mean_features) / sqrt(var_features + ε) * γ + β
    where γ, β are learned scale and shift parameters.

    Unlike BatchNorm (normalises across the batch), LayerNorm normalises each
    token's (timestep's) feature vector independently. This works better for
    sequence models because it doesn't depend on other samples in the batch.

    ATTENTION WEIGHTS STORAGE
    -------------------------
    We store self.last_attn_weights after each forward pass so we can inspect
    what the model has learned to attend to (Cell 10 — visualisation).
    Shape: (batch, n_heads, T, T) — one (T×T) attention matrix per head.
    """

    def __init__(self, d_model, n_heads, ffn_dim, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model      = d_model
        self.n_heads      = n_heads
        self.ffn_dim      = ffn_dim
        self.dropout_rate = dropout_rate

        # ── Sub-layer 1: Multi-Head Self-Attention ────────────────────────────
        # key_dim = d_k = d_model // n_heads — dimension of each head's Q/K/V projections
        # The layer internally creates separate W_Q, W_K, W_V for each head.
        self.mha    = keras.layers.MultiHeadAttention(
            num_heads=n_heads,
            key_dim=d_model // n_heads,   # each head operates in d_k dimensions
            dropout=dropout_rate,
            name='mha'
        )
        # LayerNorm for sub-layer 1 (epsilon=1e-6 avoids division by zero)
        self.norm1  = keras.layers.LayerNormalization(epsilon=1e-6, name='ln1')

        # ── Sub-layer 2: Feed-Forward Network (FFN) ───────────────────────────
        # Expands to ffn_dim then projects back to d_model.
        # Applied INDEPENDENTLY to each timestep (no cross-timestep mixing).
        self.ffn1   = keras.layers.Dense(ffn_dim, activation='relu', name='ffn1')
        self.ffn2   = keras.layers.Dense(d_model, name='ffn2')   # no activation before LayerNorm
        # LayerNorm for sub-layer 2
        self.norm2  = keras.layers.LayerNormalization(epsilon=1e-6, name='ln2')

        # ── Dropout ───────────────────────────────────────────────────────────
        # Applied inside each sub-layer (before the residual addition).
        # Dropout randomly zeros activations during training → regularisation.
        self.drop1  = keras.layers.Dropout(dropout_rate)
        self.drop2  = keras.layers.Dropout(dropout_rate)

        # Storage slot for attention weights (populated during forward pass)
        self.last_attn_weights = None   # (batch, n_heads, T, T) after each call()

    def call(self, x, training=False):
        # x: (batch, T, D_MODEL) — input sequence of token embeddings

        # ── Sub-layer 1: Multi-Head Self-Attention ────────────────────────────
        # query=x, key=x, value=x → SELF-attention (each token attends to all tokens)
        # return_attention_scores=True → also returns the (batch, heads, T, T) matrix
        attn_out, attn_weights = self.mha(
            x, x,                             # query, key (and value defaults to key)
            return_attention_scores=True,
            training=training
        )
        # Store attention weights for post-hoc visualisation (Cell 10)
        self.last_attn_weights = attn_weights  # (batch, n_heads, T, T)

        attn_out = self.drop1(attn_out, training=training)   # dropout for regularisation

        # Residual connection + LayerNorm:  x' = LN( x + attn_out )
        x = self.norm1(x + attn_out)                         # (batch, T, D_MODEL)

        # ── Sub-layer 2: Feed-Forward Network ────────────────────────────────
        # Both Dense layers are applied to each timestep independently (broadcast over T).
        ffn_out = self.ffn1(x)      # (batch, T, ffn_dim)  — expand with relu
        ffn_out = self.ffn2(ffn_out) # (batch, T, D_MODEL) — project back

        ffn_out = self.drop2(ffn_out, training=training)     # dropout

        # Residual connection + LayerNorm:  x'' = LN( x' + ffn_out )
        x = self.norm2(x + ffn_out)                          # (batch, T, D_MODEL)
        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            'd_model': self.d_model, 'n_heads': self.n_heads,
            'ffn_dim': self.ffn_dim, 'dropout_rate': self.dropout_rate
        })
        return config


# ─────────────────────────────────────────────────────────────────────────────
# 5.  Build the full model (Keras functional API)
# ─────────────────────────────────────────────────────────────────────────────
def build_transformer_model(
        n_timesteps, n_features, n_classes,
        d_model=64, n_layers=3, n_heads=4,
        ffn_dim=128, mfa_gamma=64,
        dense_units=None, dropout_rate=0.3,
        transformer_dropout=0.1,
        use_mfa=True, use_pos_enc=True,
):
    """
    Assemble the STFTnet-inspired Transformer gesture classifier.

    The Keras FUNCTIONAL API is used rather than Sequential, because the model
    has conditional blocks (optional MFA, optional positional encoding) and we
    want to inspect intermediate layer outputs later.  The functional API builds
    a computation GRAPH: you define tensors and the operations between them,
    then wrap them in a Model(inputs=..., outputs=...).

    Architecture (with shapes):
    ─────────────────────────────────────────────────────────────────────────────
    Input             : (batch, n_timesteps, n_features)   e.g. (32, 156, 180)
    [MFABlock]        : (batch, n_timesteps, n_features)   optional; gated channels
    DW-Sep-Conv1D     : (batch, n_timesteps, d_model)      embed to d_model dims
    [PosEnc]          : (batch, n_timesteps, d_model)      optional; add position signal
    TransEncoderBlock : (batch, n_timesteps, d_model)      ×n_layers; shape unchanged
    GAP               : (batch, d_model)                   aggregate time dimension
    Dense(units, relu): (batch, dense_units[0])            classification head
    Dropout           : (batch, dense_units[0])
    Dense(n_classes)  : (batch, n_classes)                 logits
    Softmax           : (batch, n_classes)                 probabilities

    Parameters
    ----------
    n_timesteps        : T — number of timesteps per trial
    n_features         : C — number of sensor channels (width of input)
    n_classes          : number of gesture classes (output units)
    d_model            : embedding / Transformer model dimension
    n_layers           : number of stacked TransformerEncoderBlocks
    n_heads            : number of attention heads (must divide d_model)
    ffn_dim            : Feed-Forward Network hidden dimension
    mfa_gamma          : MFA bottleneck dimension
    dense_units        : list of hidden units in the classification head
    dropout_rate       : dropout rate for the classification head
    transformer_dropout: dropout rate inside each encoder block
    use_mfa            : whether to include the MFA channel-attention block
    use_pos_enc        : whether to add sinusoidal positional encoding

    Returns
    -------
    keras.Model — compiled model ready for training
    """
    if dense_units is None:
        dense_units = [64]

    # ── Define the input tensor ───────────────────────────────────────────────
    # Keras functional API: start by specifying the input shape.
    # 'None' batch dimension means any batch size is valid at inference time.
    inputs = keras.Input(shape=(n_timesteps, n_features), name='input')
    x = inputs   # x tracks the "current" tensor as it flows through layers

    # ── MFA Block (optional) ──────────────────────────────────────────────────
    # Channel-attention gate: downweights uninformative sensor channels.
    # Applied before the embedding so the conv only sees relevant information.
    if use_mfa:
        # Shape: (batch, n_timesteps, n_features) → same shape, gated values
        x = MFABlock(gamma=mfa_gamma, name='mfa_block')(x)

    # ── Depthwise Separable Convolution (embedding) ───────────────────────────
    # Projects raw sensor features from n_features channels to d_model dimensions.
    # Each timestep independently becomes a d_model-dimensional token.
    # Shape: (batch, n_timesteps, n_features) → (batch, n_timesteps, d_model)
    x = DepthwiseSeparableConv1D(d_model=d_model, kernel_size=3,
                                  name='dw_sep_embed')(x)

    # ── Positional Encoding (optional) ────────────────────────────────────────
    # Adds a unique sinusoidal fingerprint to each timestep's embedding.
    # Without this, the Transformer cannot distinguish timestep 10 from timestep 100.
    # Shape: (batch, n_timesteps, d_model) → same shape (values shifted)
    if use_pos_enc:
        x = PositionalEncoding(max_len=n_timesteps + 64, name='pos_enc')(x)

    # ── Transformer Encoder Stack ─────────────────────────────────────────────
    # Stack n_layers TransformerEncoderBlocks sequentially.
    # Each block refines the token representations through self-attention and FFN.
    # Shape stays (batch, n_timesteps, d_model) through all blocks — the Transformer
    # does NOT change the sequence length.
    for i in range(n_layers):
        x = TransformerEncoderBlock(
            d_model=d_model,
            n_heads=n_heads,
            ffn_dim=ffn_dim,
            dropout_rate=transformer_dropout,
            name=f'transformer_block_{i}'   # named so we can retrieve by index later
        )(x)

    # ── Global Average Pooling — aggregate temporal context ───────────────────
    # After n_layers encoder blocks we have (batch, n_timesteps, d_model) —
    # one d_model-dimensional vector for EACH of the 156 timesteps.
    # We average across the time dimension to get a single fixed-size representation
    # for the whole sequence: (batch, n_timesteps, d_model) → (batch, d_model).
    # Alternative: take only the [CLS] token embedding (used in BERT-style models).
    x = keras.layers.GlobalAveragePooling1D(name='gap')(x)

    # ── Classification Head ───────────────────────────────────────────────────
    # Standard MLP head: Dense → Dropout → ... → Dense(n_classes, softmax)
    # Dropout during training randomly zeros some activations → prevents co-adaptation
    # of head neurons → reduces overfitting on the small (~280 sample) training set.
    for i, units in enumerate(dense_units):
        x = keras.layers.Dense(units, activation='relu',
                                name=f'head_dense_{i}')(x)         # (batch, units)
        x = keras.layers.Dropout(dropout_rate, name=f'head_drop_{i}')(x)

    # Output layer: softmax converts raw scores (logits) to probabilities.
    # outputs[i] = probability that the input belongs to gesture class i.
    # They sum to 1.0 across the n_classes dimension.
    outputs = keras.layers.Dense(n_classes, activation='softmax',
                                  name='output')(x)                 # (batch, n_classes)

    # Wrap inputs → outputs in a named Model
    model = keras.Model(inputs=inputs, outputs=outputs, name='STFTnet_Transformer')
    return model


# ── Instantiate the model ──────────────────────────────────────────────────────
# Use the shape variables derived from the loaded dataset (Cell 5).
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

model = build_transformer_model(
    n_timesteps         = N_TIMESTEPS,
    n_features          = N_FEATURES,
    n_classes           = N_CLASSES,
    d_model             = D_MODEL,
    n_layers            = N_LAYERS,
    n_heads             = N_HEADS_ACTUAL,
    ffn_dim             = FFN_DIM,
    mfa_gamma           = MFA_GAMMA,
    dense_units         = DENSE_UNITS,
    dropout_rate        = DROPOUT_RATE,
    transformer_dropout = TRANSFORMER_DROPOUT,
    use_mfa             = USE_MFA_BLOCK,
    use_pos_enc         = USE_POSITIONAL_ENCODING,
)

# ── Optimiser ────────────────────────────────────────────────────────────────
# AdamW = Adam + weight decay (L2 regularisation applied directly to weights,
# not through the gradient — more principled than adding L2 to the loss).
# If tensorflow-addons is unavailable, fall back to standard Adam.
if TFA_AVAILABLE:
    optimizer = tfa.optimizers.AdamW(
        learning_rate = LEARNING_RATE,
        weight_decay  = WEIGHT_DECAY,   # penalises large weights → reduces overfitting
    )
    print(f"Optimizer: AdamW (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
else:
    optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    print(f"Optimizer: Adam (lr={LEARNING_RATE})  — weight_decay={WEIGHT_DECAY} not applied")

# Compile: attach the loss function and metrics.
# sparse_categorical_crossentropy: expects INTEGER labels (0, 1, 2, 3) rather than
# one-hot encoded vectors. This avoids having to call to_categorical().
model.compile(
    optimizer = optimizer,
    loss      = 'sparse_categorical_crossentropy',
    metrics   = ['accuracy'],
)

# Print a summary of every layer, its output shape, and its parameter count
model.summary()

---

## Cell 8 — Training with Callbacks and Learning Curves

### Learning rate warmup

The learning rate starts near zero and **ramps up linearly** over `LR_WARMUP_STEPS`
gradient steps before settling at `LEARNING_RATE`. Why?

At the start of training, all weights are randomly initialised — the model's
representations are meaningless. If we apply the full learning rate immediately,
the large gradient updates can push weights into bad regions from which it is hard
to recover. The warmup period lets the model establish rough internal representations
at a slow pace before making large committed updates.

### Callbacks

| Callback | Purpose |
|---|---|
| `ModelCheckpoint` | Save the model weights whenever `val_accuracy` improves |
| `EarlyStopping` | Stop training if `val_accuracy` hasn't improved for `EARLY_STOP_PATIENCE` epochs; restore the best weights |
| `ReduceLROnPlateau` | Halve the learning rate if `val_loss` plateaus for 7 epochs |
| `CSVLogger` | Append training metrics to a CSV file after every epoch |
| `WarmupCosineSchedule` | Perform the linear LR warmup |

In [ ]:
"""
Train the Transformer model with learning rate warmup, checkpointing,
early stopping, and automatic learning rate reduction on plateau.

After training we plot two learning curves:
  Loss curve    — train vs. val sparse categorical cross-entropy over epochs
  Accuracy curve — train vs. val accuracy over epochs
These are the standard curves you would examine to diagnose overfitting or underfitting:
  - Train >> Val  → overfitting (model memorised training set)
  - Both low      → underfitting (model not expressive enough, or not enough training)
  - Both high and close → good generalisation
"""

# ── Paths ─────────────────────────────────────────────────────────────────────
RESULTS_DIR      = Path('results')
SAVED_MODELS_DIR = Path('saved_models')
RESULTS_DIR.mkdir(exist_ok=True)        # create directories if they don't already exist
SAVED_MODELS_DIR.mkdir(exist_ok=True)

# Best-checkpoint file (Keras native format, saves weights + architecture)
CKPT_PATH = SAVED_MODELS_DIR / 'transformer_best.keras'


# ── LR Warmup callback ────────────────────────────────────────────────────────
class WarmupCosineSchedule(keras.callbacks.Callback):
    """
    Linear learning rate warmup for the first LR_WARMUP_STEPS gradient steps.

    Problem it solves:
        At step 0, all model weights are random. The gradient of the loss
        w.r.t. these random weights can be large and point in unreliable directions.
        Applying the full LEARNING_RATE immediately causes large weight updates
        that can push the model into a poor local minimum or cause the loss to spike.

    Solution:
        Start with lr ≈ 0. On each gradient step (each training batch), increase lr
        linearly: lr(step) = LEARNING_RATE * (step / LR_WARMUP_STEPS).
        After LR_WARMUP_STEPS steps, lr = LEARNING_RATE and the warmup ends.

    LR_WARMUP_STEPS = 50 means the warmup covers 50 / (280/32) ≈ 5-6 epochs
    (280 training samples / batch_size=32 ≈ 9 steps per epoch).

    Implementation: Keras callbacks expose hooks like on_train_batch_begin(),
    which fires at the start of every gradient step. We use it to directly set
    the optimizer's learning rate via the backend K.set_value().
    """
    def __init__(self, warmup_steps, base_lr):
        super().__init__()
        self.warmup_steps = warmup_steps   # total warmup gradient steps
        self.base_lr      = base_lr        # target LR after warmup
        self.step         = 0              # global step counter

    def on_train_batch_begin(self, batch, logs=None):
        if self.warmup_steps <= 0:
            return   # warmup disabled — leave LR unchanged
        self.step += 1
        if self.step <= self.warmup_steps:
            # Linearly ramp from near-zero to base_lr
            lr = self.base_lr * (self.step / self.warmup_steps)
            K.set_value(self.model.optimizer.lr, lr)


# ── Callbacks ─────────────────────────────────────────────────────────────────
callbacks = [
    # Save the best model weights (by val_accuracy) to disk
    keras.callbacks.ModelCheckpoint(
        filepath        = str(CKPT_PATH),
        monitor         = 'val_accuracy',   # the metric to watch
        save_best_only  = True,             # only save if this epoch is better
        save_format     = 'keras',
        verbose         = 1,
    ),
    # Stop early if val_accuracy stops improving (prevents wasted compute and overfitting)
    keras.callbacks.EarlyStopping(
        monitor              = 'val_accuracy',
        patience             = EARLY_STOP_PATIENCE,   # epochs of no improvement before stopping
        restore_best_weights = True,   # revert to the best weights when stopping
        verbose              = 1,
    ),
    # Halve the LR if val_loss plateaus — helps escape shallow local minima
    keras.callbacks.ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.5,      # new_lr = old_lr * 0.5
        patience = 7,        # epochs of no improvement before reducing
        min_lr   = 1e-6,     # lower bound — LR won't be reduced below this
        verbose  = 1,
    ),
    # Write a CSV log of all metrics after each epoch (useful for offline analysis)
    keras.callbacks.CSVLogger(
        str(RESULTS_DIR / 'transformer_training_log.csv')
    ),
]

# Add warmup callback only if LR_WARMUP_STEPS > 0
if LR_WARMUP_STEPS > 0:
    callbacks.append(WarmupCosineSchedule(LR_WARMUP_STEPS, LEARNING_RATE))
    print(f"LR warmup enabled: {LR_WARMUP_STEPS} steps")

# ── Train ─────────────────────────────────────────────────────────────────────
print(f"\nTraining for up to {EPOCHS} epochs ")
print(f"  Batch size: {BATCH_SIZE}  |  Early stop patience: {EARLY_STOP_PATIENCE}")

history = model.fit(
    X_train, y_train,
    validation_data = (X_val, y_val),   # validation set evaluated after each epoch
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    callbacks       = callbacks,
    verbose         = 1,
)

# ── Learning Curves ───────────────────────────────────────────────────────────
# Plotting both splits (train and val) for both metrics (loss and accuracy)
# lets you visually diagnose overfitting:
#   Large gap between train and val → overfitting
#   Both curves flat and low → underfitting
hist      = history.history
epochs_ran = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# ── Loss curve ────────────────────────────────────────────────────────────────
axes[0].plot(epochs_ran, hist['loss'],     color=TEAL, lw=2, label='Train loss')
axes[0].plot(epochs_ran, hist['val_loss'], color=RUST, lw=2, label='Val loss', linestyle='--')
axes[0].set_title('Loss (Sparse Categorical Cross-Entropy)', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# ── Accuracy curve ────────────────────────────────────────────────────────────
axes[1].plot(epochs_ran, hist['accuracy'],     color=TEAL, lw=2, label='Train acc')
axes[1].plot(epochs_ran, hist['val_accuracy'], color=RUST, lw=2, label='Val acc', linestyle='--')
axes[1].set_title('Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim([0, 1.05])
axes[1].legend()
axes[1].grid(True)

# Annotate the best val_accuracy epoch
best_val_acc = max(hist['val_accuracy'])
best_ep      = hist['val_accuracy'].index(best_val_acc) + 1   # 1-indexed epoch

# Vertical dashed line at the best epoch
axes[1].axvline(best_ep, color=DARK_TEAL, lw=1, linestyle=':')
axes[1].text(best_ep + 0.3, best_val_acc - 0.04,
             f'Best={best_val_acc:.4f}\n(ep {best_ep})',
             fontsize=8, color=DARK_TEAL)

plt.suptitle('Training Curves — STFTnet Transformer', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'transformer_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nBest val accuracy : {best_val_acc:.4f}  (epoch {best_ep})")

---

## Cell 9 — Test Set Evaluation

After training, we evaluate the **best checkpoint** (saved by ModelCheckpoint)
on the held-out test set. The test set was NEVER seen during training or used
by any callback — it gives an **unbiased estimate of generalisation performance**.

### Metrics explained (from your COMP3308 lectures)

**Accuracy**: proportion of correctly classified test samples.
Appropriate here because the dataset is balanced (equal samples per class).

**Precision** (per class): of all samples predicted as class C, how many
actually were class C? Measures false positives.

**Recall** (per class): of all samples actually in class C, how many were
correctly identified? Measures false negatives.

**F1** (per class): harmonic mean of precision and recall.
`F1 = 2 × precision × recall / (precision + recall)`

**Confusion matrix**: rows = true class, columns = predicted class.
The diagonal entries are correct predictions. Off-diagonal entries reveal
which gesture classes are confused with which others.

In [ ]:
"""
Evaluate the best saved model on the held-out test set.

Steps:
  1. Reload the best checkpoint (weights that achieved the highest val_accuracy)
  2. Run model.evaluate() to get test loss and accuracy
  3. Run model.predict() to get class probabilities for each test sample
  4. Convert probabilities to class predictions: y_pred = argmax(y_prob, axis=1)
  5. Compute classification report (precision, recall, F1 per class)
  6. Plot the confusion matrix (raw counts and normalised by true class)
"""

# ── Reload best checkpoint ────────────────────────────────────────────────────
# The ModelCheckpoint callback saved the best weights to CKPT_PATH.
# We reload them with custom_objects so Keras knows how to deserialise the
# custom layer classes defined in Cell 7.
if CKPT_PATH.exists():
    model = keras.models.load_model(
        str(CKPT_PATH),
        custom_objects={
            'MFABlock':                  MFABlock,
            'DepthwiseSeparableConv1D':  DepthwiseSeparableConv1D,
            'PositionalEncoding':         PositionalEncoding,
            'TransformerEncoderBlock':   TransformerEncoderBlock,
        }
    )
    print("Loaded best checkpoint.")
else:
    print("No checkpoint found — using current (in-memory) weights.")

# ── Test set evaluation ───────────────────────────────────────────────────────
# model.evaluate() runs a forward pass on every test sample (no gradient updates)
# and computes the compiled metrics (loss and accuracy).
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)")

# ── Predictions ───────────────────────────────────────────────────────────────
# model.predict() returns a (N_test, N_CLASSES) probability array.
# y_prob[i, c] = probability that sample i belongs to class c.
# argmax gives the class with the highest probability — the model's prediction.
y_prob = model.predict(X_test, verbose=0)     # (N_test, N_CLASSES) — softmax probs
y_pred = np.argmax(y_prob, axis=1)            # (N_test,)           — integer predictions

# ── Classification Report ─────────────────────────────────────────────────────
# sklearn's classification_report prints precision, recall, F1, and support
# (number of true instances) for each class individually, plus macro and
# weighted averages across classes.
print("\n" + "=" * 60)
print("Classification Report")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=labels, digits=4))

# ── Confusion Matrix ──────────────────────────────────────────────────────────
# cm[i, j] = number of samples with true label i that were predicted as label j.
# Diagonal entries (i=j) are correct; off-diagonal are errors.
cm      = confusion_matrix(y_test, y_pred)
# Normalise by true class (divide each row by its sum):
# cm_norm[i, j] = fraction of class i examples predicted as j.
# Allows fair comparison across classes even if class sizes differ.
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(max(12, N_CLASSES * 0.9), max(8, N_CLASSES * 0.7)))

# Left: raw integer counts
sns.heatmap(
    cm, ax=axes[0],
    annot=(N_CLASSES <= 20), fmt='d', cmap='YlOrRd',
    xticklabels=labels, yticklabels=labels,
    linewidths=0.5, cbar=True
)
axes[0].set_title('Confusion Matrix (counts)', fontweight='bold')
axes[0].set_xlabel('Predicted label')
axes[0].set_ylabel('True label')
axes[0].tick_params(axis='x', rotation=45)

# Right: normalised (shows per-class recall on the diagonal)
sns.heatmap(
    cm_norm, ax=axes[1],
    annot=(N_CLASSES <= 20), fmt='.2f', cmap='YlOrBr',
    xticklabels=labels, yticklabels=labels,
    vmin=0, vmax=1, linewidths=0.5, cbar=True
)
axes[1].set_title('Confusion Matrix (normalised by true class)', fontweight='bold')
axes[1].set_xlabel('Predicted label')
axes[1].set_ylabel('True label')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle(f'Test Accuracy: {test_acc:.4f}', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'transformer_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Per-class accuracy summary ────────────────────────────────────────────────
# Diagonal of the normalised confusion matrix = per-class accuracy (= recall).
# Identifies which gestures are hardest for the model.
per_class_acc = cm.diagonal() / cm.sum(axis=1)
worst_classes = np.argsort(per_class_acc)[:5]   # indices of the 5 worst classes
print("\nLowest per-class accuracy:")
for idx in worst_classes:
    print(f"  {labels[idx]:>50s}  {per_class_acc[idx]:.4f}")

---

## Cell 10 — Attention Weight Visualisation

One of the key interpretability advantages of Transformers over LSTMs is that
**attention weights are directly inspectable**.

Each TransformerEncoderBlock computes a **(T × T) attention matrix** per head,
where `T = N_TIMESTEPS = 156`. Entry `[i, j]` is the weight that timestep *i*
assigns to timestep *j* when building its contextual representation:
- A weight of 1.0 at `[i, j]` means timestep *i* focuses entirely on timestep *j*
- Weights sum to 1.0 across each row *i* (softmax normalisation)

### Reading the heatmaps

- **Bright diagonal** → the model mostly attends locally (each timestep looks at itself
  and its immediate neighbours — similar to a convolution)
- **Off-diagonal bright patches** → the model relates distant timesteps — e.g. the
  gesture's opening posture to its closing posture
- **Different heads** typically specialise in different patterns (local vs. global;
  different gesture phases)

We plot the **last encoder block's** attention because it operates on the highest-level
abstract representations — the most semantically meaningful level.

We also show:
1. **Per-head attention maps** — see what different heads specialise in
2. **Attention received per timestep** (column sums) — which moments are most used as context
3. **Attention sent per timestep** (row sums) — which moments look most broadly
4. **Block-by-block mean attention evolution** — how attention patterns change through the network

In [ ]:
"""
Visualise self-attention weights from the trained Transformer.

Strategy:
  - Each TransformerEncoderBlock stores its last attention weights in
    self.last_attn_weights (set during the forward pass in call()).
  - We run one forward pass on a single test sample, then read out these
    stored weights from each block.
  - We plot per-head heatmaps, mean attention, and temporal profiles.
"""

# ── Identify the TransformerEncoderBlock layers ────────────────────────────────
# model.layers is a list of all layers in the computation graph.
# We filter for instances of our custom class.
def get_attention_extractor(model):
    """
    Return a list of all TransformerEncoderBlock layers in the model.
    These are the layers that store attention weights in .last_attn_weights.
    """
    transformer_blocks = [
        layer for layer in model.layers
        if isinstance(layer, TransformerEncoderBlock)
    ]
    if not transformer_blocks:
        raise RuntimeError("No TransformerEncoderBlock found in model.")
    return transformer_blocks

transformer_blocks = get_attention_extractor(model)
print(f"Found {len(transformer_blocks)} TransformerEncoderBlock(s) in model.")

# ── Select a test sample for visualisation ────────────────────────────────────
# Use the first CORRECTLY classified test sample — ensures we're visualising
# what the model learned, not how it handles a failure case.
correct_mask      = (y_pred == y_test)
first_correct_idx = np.where(correct_mask)[0][0] if correct_mask.any() else 0

viz_sample  = X_test[first_correct_idx:first_correct_idx+1]   # (1, T, C) — batch of 1
true_label  = labels[y_test[first_correct_idx]]
pred_label  = labels[y_pred[first_correct_idx]]

print(f"\nVisualisation sample:  true='{true_label}',  predicted='{pred_label}'")


# ── Extract attention weights via a forward pass ──────────────────────────────
def extract_attention_weights(model, x, block_idx=None):
    """
    Run a single forward pass and collect attention weights from all encoder blocks.

    The TransformerEncoderBlock.call() method stores attention weights in
    self.last_attn_weights after each forward pass:
        shape: (batch, n_heads, T, T)

    We return a list of (n_heads, T, T) arrays — one per encoder block.
    The list is ordered from block 0 (first, lowest-level) to block N-1 (last).

    Parameters
    ----------
    model     : trained Keras model
    x         : input tensor of shape (1, T, C) — single sample
    block_idx : if specified, return only that block's weights

    Returns
    -------
    list of numpy arrays, each of shape (n_heads, T, T)
    """
    blocks = [layer for layer in model.layers
              if isinstance(layer, TransformerEncoderBlock)]

    # Run a forward pass with training=False (disables dropout for clean attention)
    _ = model(x, training=False)

    attn_list = []
    for blk in blocks:
        if blk.last_attn_weights is not None:
            w = blk.last_attn_weights.numpy()   # (1, n_heads, T, T) → convert to numpy
            attn_list.append(w[0])               # (n_heads, T, T) — remove batch dim
    return attn_list


attn_weights_list = extract_attention_weights(model, viz_sample)
n_blocks = len(attn_weights_list)
print(f"Extracted attention weights from {n_blocks} block(s).")
if n_blocks > 0:
    print(f"Shape per block: {attn_weights_list[0].shape}  (heads × T × T)")


# ── Figure 1: Mean attention map + per-head grid ──────────────────────────────
last_attn = attn_weights_list[-1]        # (n_heads, T, T) from the LAST encoder block
n_heads_actual = last_attn.shape[0]

# Mean over all heads: what does the model attend to on average?
mean_attn = last_attn.mean(axis=0)      # (T, T)

ncols = min(n_heads_actual, 4)
nrows = 1 + int(np.ceil(n_heads_actual / ncols))

fig = plt.figure(figsize=(4 * ncols, 3.5 * nrows))
gs  = gridspec.GridSpec(nrows, ncols, figure=fig)

# Top row: mean attention heatmap
ax_mean = fig.add_subplot(gs[0, :ncols])
im = ax_mean.imshow(mean_attn, cmap='viridis', aspect='auto', interpolation='nearest')
plt.colorbar(im, ax=ax_mean, fraction=0.02)
ax_mean.set_title(
    f'Mean Attention (across {n_heads_actual} heads) — Block {n_blocks}\n'
    f'Sample: "{true_label}"',
    fontweight='bold', fontsize=11
)
ax_mean.set_xlabel('Key timestep j (source of information)')
ax_mean.set_ylabel('Query timestep i (consumer of information)')

# Mark the single highest-weight (i, j) pair in the mean attention map
peak_q, peak_k = np.unravel_index(mean_attn.argmax(), mean_attn.shape)
ax_mean.scatter(peak_k, peak_q, color='red', s=60, zorder=5, label='Peak attention')
ax_mean.legend(fontsize=8, loc='upper right')

# Individual per-head maps (shows specialisation across heads)
for h in range(n_heads_actual):
    row = 1 + h // ncols
    col = h %  ncols
    ax  = fig.add_subplot(gs[row, col])
    ax.imshow(last_attn[h], cmap='plasma', aspect='auto', interpolation='nearest')
    ax.set_title(f'Head {h+1}', fontsize=9)
    ax.set_xlabel('Key t', fontsize=7)
    ax.set_ylabel('Query t', fontsize=7)
    ax.tick_params(labelsize=6)

plt.suptitle(
    f'Self-Attention Weights — Last Transformer Block\n'
    f'Gesture: "{true_label}"  |  Sequence length: {N_TIMESTEPS} steps',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'transformer_attention_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


# ── Figure 2: Temporal attention profile ─────────────────────────────────────
# Summarise the (T×T) attention map into two 1-D curves:
#
#  Attention RECEIVED at timestep j  = sum over all i of α(i,j)  [column sum]
#    → How important is timestep j as a source of context?
#    → High value: many other timesteps look here for information.
#
#  Attention SENT from timestep i    = sum over all j of α(i,j)  [row sum]
#    → How broadly does timestep i cast its attention?
#    → High value: this timestep draws information from many others.
attn_received = mean_attn.sum(axis=0)   # (T,) — column sums
attn_sent     = mean_attn.sum(axis=1)   # (T,) — row sums

# Convert timestep index to milliseconds for a physically meaningful x-axis
time_axis_ms  = np.arange(N_TIMESTEPS) / FS_HZ * 1000   # ms

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

ax1.fill_between(time_axis_ms, attn_received, color=TEAL, alpha=0.7, label='Attention received')
ax1.set_ylabel('Attention weight (sum)')
ax1.set_title('Attention Received — how much each timestep is used as context by others',
              fontsize=10, fontweight='bold')
# Mark the peak: the single most attended-to moment in the gesture
ax1.axvline(time_axis_ms[attn_received.argmax()], color=RUST, lw=1.5, linestyle='--')
ax1.text(time_axis_ms[attn_received.argmax()], attn_received.max() * 0.9,
         f'{time_axis_ms[attn_received.argmax()]:.0f} ms', fontsize=8, color=RUST)
ax1.legend(fontsize=9)
ax1.grid(True)

ax2.fill_between(time_axis_ms, attn_sent, color=RUST, alpha=0.7, label='Attention sent')
ax2.set_xlabel('Time (ms)')
ax2.set_ylabel('Attention weight (sum)')
ax2.set_title('Attention Sent — how broadly each timestep draws from others',
              fontsize=10, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True)

plt.suptitle(f'Temporal Attention Profile — "{true_label}"', fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'transformer_temporal_attention.png', dpi=150, bbox_inches='tight')
plt.show()


# ── Figure 3: Attention evolution across encoder blocks ───────────────────────
# Show how the mean attention pattern changes from block 0 (low-level) to
# block N-1 (high-level abstract representations).
# Early blocks tend to show local attention (bright diagonal);
# later blocks often develop more global/distributed attention patterns.
if n_blocks > 1:
    fig, axes = plt.subplots(1, n_blocks, figsize=(5 * n_blocks, 4.5))
    if n_blocks == 1:
        axes = [axes]

    for bi, (attn_blk, ax) in enumerate(zip(attn_weights_list, axes)):
        m  = attn_blk.mean(axis=0)   # (T, T) — mean over heads for this block
        im = ax.imshow(m, cmap='viridis', aspect='auto', interpolation='nearest')
        ax.set_title(f'Block {bi+1} — mean across heads', fontweight='bold')
        ax.set_xlabel('Key timestep j')
        ax.set_ylabel('Query timestep i')
        plt.colorbar(im, ax=ax, fraction=0.04)

    plt.suptitle(
        f'Mean Attention per Encoder Block — "{true_label}"\n'
        f'(Block 1 = lowest-level, Block {n_blocks} = highest-level)',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    fig.savefig(RESULTS_DIR / 'transformer_block_attention_evolution.png',
                dpi=150, bbox_inches='tight')
    plt.show()

print("Attention visualisations saved to results/.")

---

## Cell 11 — Save Model and Results

Saves:
1. **Keras native format** (`.keras`) — full model including architecture, weights,
   and optimiser state; reloadable with `keras.models.load_model()`
2. **SavedModel format** — TensorFlow's portable format compatible with TF Serving,
   TF Lite conversion, and deployment on other platforms
3. **Results JSON** — all configuration, dataset statistics, and final metrics in
   a structured dictionary; used by `00_Results_Comparison.ipynb` to compare models

In [ ]:
"""
Compute final metrics on all three splits, save the trained model in two
formats, and write a structured JSON results file for later comparison.

The results JSON captures:
  - The full CONFIG (so you can reproduce any experiment)
  - Dataset statistics (N, T, C, class names)
  - Final metrics (loss and accuracy on train/val/test)
  - Complete epoch-by-epoch training history
  - Paths to all saved artefacts
"""

import datetime

# ── Final metrics on all three splits ─────────────────────────────────────────
# Evaluate on train, val, AND test to check for overfitting.
# A large gap between train_acc and test_acc signals the model memorised training data.
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
val_loss,   val_acc   = model.evaluate(X_val,   y_val,   verbose=0)
test_loss,  test_acc  = model.evaluate(X_test,  y_test,  verbose=0)

print(f"Train : loss={train_loss:.4f},  acc={train_acc:.4f}")
print(f"Val   : loss={val_loss:.4f},  acc={val_acc:.4f}")
print(f"Test  : loss={test_loss:.4f},  acc={test_acc:.4f}")

# ── Full model save (Keras format) ────────────────────────────────────────────
# Saves the entire model: architecture graph, weights, and optimiser state.
# Required for resuming training or for exact reproduction.
final_model_path = SAVED_MODELS_DIR / 'transformer_final.keras'
model.save(str(final_model_path))
print(f"\nFull model saved to: {final_model_path}")

# ── SavedModel export (TFLite / TF Serving compatible) ────────────────────────
# Exports the model as a SavedModel directory — required for deployment to
# Android/iOS (via TFLite conversion) or to a TF Serving inference server.
saved_model_path = SAVED_MODELS_DIR / 'transformer_savedmodel'
model.export(str(saved_model_path))
print(f"SavedModel exported to: {saved_model_path}")

# ── Results JSON ──────────────────────────────────────────────────────────────
# Build a nested dictionary capturing everything needed to compare this run
# against other notebook results (LSTM, CNN, CNN-BiLSTM-Attention, etc.).
from sklearn.metrics import classification_report as skl_cr
report_dict = skl_cr(y_test, y_pred, target_names=labels, output_dict=True)

results = {
    "model": "STFTnet_Transformer",
    "timestamp": datetime.datetime.now().isoformat(),

    # Full configuration snapshot (all CONFIG variables)
    "config": {
        "data_root":             DATA_ROOT,
        "filter_type":           FILTER_TYPE,
        "target_len":            TARGET_LEN,
        "normalization":         NORMALIZATION,
        "per_sample_norm":       PER_SAMPLE_NORM,
        "use_mfa_block":         USE_MFA_BLOCK,
        "mfa_gamma":             MFA_GAMMA,
        "d_model":               D_MODEL,
        "n_layers":              N_LAYERS,
        "n_heads_requested":     N_HEADS,
        "n_heads_actual":        N_HEADS_ACTUAL,
        "ffn_dim":               FFN_DIM,
        "transformer_dropout":   TRANSFORMER_DROPOUT,
        "dense_units":           DENSE_UNITS,
        "dropout_rate":          DROPOUT_RATE,
        "epochs":                EPOCHS,
        "batch_size":            BATCH_SIZE,
        "learning_rate":         LEARNING_RATE,
        "weight_decay":          WEIGHT_DECAY,
        "early_stop_patience":   EARLY_STOP_PATIENCE,
        "lr_warmup_steps":       LR_WARMUP_STEPS,
        "use_positional_encoding": USE_POSITIONAL_ENCODING,
        "optimizer":             "AdamW" if TFA_AVAILABLE else "Adam",
        "train_ratio":           TRAIN_RATIO,
        "val_ratio":             VAL_RATIO,
        "random_seed":           RANDOM_SEED,
    },

    # Dataset statistics
    "dataset": {
        "n_samples":   int(X.shape[0]),
        "n_timesteps": int(X.shape[1]),
        "n_features":  int(X.shape[2]),
        "n_classes":   int(N_CLASSES),
        "classes":     labels,
        "n_train":     int(len(X_train)),
        "n_val":       int(len(X_val)),
        "n_test":      int(len(X_test)),
    },

    # Summary metrics
    "metrics": {
        "train_loss":       float(train_loss),
        "train_accuracy":   float(train_acc),
        "val_loss":         float(val_loss),
        "val_accuracy":     float(val_acc),
        "test_loss":        float(test_loss),
        "test_accuracy":    float(test_acc),
        "best_val_accuracy": float(best_val_acc),
        "best_val_epoch":   int(best_ep),
        "epochs_trained":   int(len(history.history['loss'])),
    },

    # Per-class precision / recall / F1
    "classification_report": report_dict,

    # Full epoch-by-epoch learning curves (used by 00_Results_Comparison.ipynb)
    "training_history": {
        "loss":         [float(v) for v in history.history['loss']],
        "accuracy":     [float(v) for v in history.history['accuracy']],
        "val_loss":     [float(v) for v in history.history['val_loss']],
        "val_accuracy": [float(v) for v in history.history['val_accuracy']],
    },

    # Paths to saved artefacts
    "artifacts": {
        "model_keras":         str(final_model_path),
        "model_savedmodel":    str(saved_model_path),
        "training_curves":     str(RESULTS_DIR / 'transformer_training_curves.png'),
        "confusion_matrix":    str(RESULTS_DIR / 'transformer_confusion_matrix.png'),
        "attention_heatmap":   str(RESULTS_DIR / 'transformer_attention_heatmap.png'),
        "temporal_attention":  str(RESULTS_DIR / 'transformer_temporal_attention.png'),
        "training_log":        str(RESULTS_DIR / 'transformer_training_log.csv'),
    }
}

results_path = RESULTS_DIR / 'transformer_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nResults JSON saved to: {results_path}")

# ── Final summary table ───────────────────────────────────────────────────────
print("\n" + "═" * 60)
print("  FINAL RESULTS SUMMARY")
print("═" * 60)
print(f"  Model         : STFTnet-inspired Transformer")
print(f"  D_MODEL       : {D_MODEL}  |  N_LAYERS: {N_LAYERS}  |  N_HEADS: {N_HEADS_ACTUAL}")
print(f"  MFA Block     : {'Yes' if USE_MFA_BLOCK else 'No'}")
print(f"  Pos Encoding  : {'Yes' if USE_POSITIONAL_ENCODING else 'No'}")
print(f"  Features      : {N_FEATURES}  |  Timesteps: {N_TIMESTEPS}")
print(f"  Classes       : {N_CLASSES}  ({CLASS_NAMES})")
print(f"  ─────────────────────────────────────────────────────────")
print(f"  Train Accuracy: {train_acc*100:.2f}%")
print(f"  Val   Accuracy: {val_acc*100:.2f}%")
print(f"  Test  Accuracy: {test_acc*100:.2f}%")
print("═" * 60)